# 03 — HFBTP Baseline Regression Models

**Purpose:** the initial HFBTP baseline workflow, implemented strictly per the approved evaluation protocol:

- **Registered splits:** **T2/D0** is the primary draw of the headline unseen-workload experiment (test levels {30,140}, validation {150,160}, training capped at level 145). **T1** (interior block-size band, test [210..305]) is the secondary block-size transfer; **T1b** is the right-edge block-size extrapolation within the measured grid (test band **[710..800]**, validation [610..705], training capped at block size **605**); **T3** (unseen orderers {5,9}) is exploratory. All four are reported below. The detailed plots in Sections 3–5 show T1b only, which does not make T1b the primary split. The protocol's headline summary is the range across draws D0–D3; D1–D3 are not implemented in this repository.
- **Targets:** `Throughput` (tx/s), `Avg Latency` (s). **Features:** only pre-execution columns {Set Transaction Arrival Rate, Block Size, Orderers} + documented decision-time-safe derivations (log terms, block-per-load ratio, orderer interactions).
- **Models:** DummyRegressor (median), LinearRegression (fitted on the eight engineered features and reported as Multiple Linear Regression (8 Engineered Features); model key `linear`), RandomForestRegressor — the transparent ladder authorized for this phase.

**Evaluation rule of the house:** a high validation/test R² on a nearly deterministic response surface is *not* evidence of practical controller benefit; MAE is reported first and interpreted in physical units. No claim beyond the measured grid.

## Setup: dataset, features, seeds, versions

In [1]:
from pathlib import Path
import sys, json, platform
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sklearn

ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from fabric_perf.data.hfbtp import HFBTPDatasetLoader
from fabric_perf.evaluation.splitters import t1_interior_band_split, t1b_right_edge_band_split, t2_d0_workload_split, t3_orderer_split
from fabric_perf.models import add_engineered_features, run_all_baselines, FEATURE_COLUMNS, RAW_FEATURES, TARGETS
from fabric_perf.utils import seed_everything
from fabric_perf.utils.helpers import figures_dir, results_dir, save_json, save_csv_rows, utc_timestamp

SEED = seed_everything(42)
FIGS = figures_dir(); RES = results_dir()
loader = HFBTPDatasetLoader(ROOT / "data" / "raw" / "hfbtp")
check = loader.compute_checksums()
df = add_engineered_features(loader.load_raw())
print("dataset md5.....:", check['md5'])
print("features........:", FEATURE_COLUMNS)
print("targets.........:", TARGETS)
print("python..........:", platform.python_version(), "| sklearn:", sklearn.__version__, "| pandas:", pd.__version__, "| numpy:", np.__version__)

dataset md5.....: E9275C49A2D7DC1F2A4015B6D8B28482
features........: ['Set Transaction Arrival Rate', 'Block Size', 'Orderers', 'log_set_rate', 'log_block_size', 'block_per_unit_load', 'set_x_orderers', 'block_over_orderers']
targets.........: ['Throughput', 'Avg Latency']
python..........: 3.14.3 | sklearn: 1.9.1 | pandas: 3.0.6 | numpy: 2.5.4


## 1. Registered splits — protocol constants, verified at construction

In [2]:
splits = {}
for name, fn in [('T1_interior', t1_interior_band_split), ('T1b_right_edge', t1b_right_edge_band_split), ('T2_D0_workload', t2_d0_workload_split), ('T3_orderer_transfer', t3_orderer_split)]:
    tr, va, te, info = fn(df)
    splits[name] = (tr, va, te, info)
    print(name, '->', 'train/val/test:', info['train_rows'], info.get('val_rows'), info.get('test_rows'), '| key bounds:', info.get('max_trained_block_size', info.get('max_trained_workload_level','-')))
# Protocol reference values (approved): T1 = 18,486 / 3,120 / 3,081; T2 D0 = 16,458 / 1,266 / 1,266 (max trained level 145); T3 = 11,766 / 636 / 12,285
assert splits['T1_interior'][3]['train_rows'] == 18486
assert splits['T2_D0_workload'][3]['train_rows'] == 16458 and splits['T2_D0_workload'][3]['max_trained_workload_level'] == 145
assert splits['T3_orderer_transfer'][3]['train_rows'] == 11766
assert splits['T1b_right_edge'][3]['max_trained_block_size'] == 605
print('All protocol reference counts verified.')

T1_interior -> train/val/test: 18486 3120 3081 | key bounds: 800
T1b_right_edge -> train/val/test: 18642 3120 2925 | key bounds: 605
T2_D0_workload -> train/val/test: 16458 1266 1266 | key bounds: 145
T3_orderer_transfer -> train/val/test: 11766 636 12285 | key bounds: -
All protocol reference counts verified.


## 2. Run the authorized baseline ladder on each split

**Model keys.** The tables and plots below label models by stable identifiers, not display names: `dummy_median` is the Dummy Regressor (median); `linear` is **Multiple Linear Regression (8 Engineered Features)**, a `StandardScaler` followed by `LinearRegression` on the eight engineered features (not polynomial regression, and not the frozen Phase 1C three-feature baseline, which is Multiple Linear Regression (3 Raw Features, Frozen Phase 1C)); `random_forest` is the Random Forest Regressor. Display names are defined in `fabric_perf.models.MODEL_LABELS`. Stored outputs below are those of the original run; this note and the model names in prose were edited afterwards without re-executing any cell.

In [3]:
all_rows = []
for name, (tr, va, te, info) in splits.items():
    rows = run_all_baselines(tr, va, te)
    for r in rows:
        r['split'] = name
        all_rows.append(r)
    print(f'{name:18s} done ({len(rows)} model-target runs)')
results_df = pd.DataFrame(all_rows)
cols = ['split','target','unit','model','train_rows','val_rows','test_rows','validation_mae','validation_rmse','validation_r2','test_mae','test_rmse','test_r2']
results_df[cols].round(3)

T1_interior        done (6 model-target runs)


T1b_right_edge     done (6 model-target runs)


T2_D0_workload     done (6 model-target runs)


T3_orderer_transfer done (6 model-target runs)


,split,target,unit,model,train_rows,val_rows,test_rows,validation_mae,validation_rmse,validation_r2,test_mae,test_rmse,test_r2
0,T1_interior,Throughput,tx/s,dummy_median,18486,3120,3081,40.165,47.146,-0.005,40.119,47.073,-0.005
1,T1_interior,Throughput,tx/s,linear,18486,3120,3081,5.026,7.036,0.978,4.874,7.131,0.977
2,T1_interior,Throughput,tx/s,random_forest,18486,3120,3081,1.316,3.763,0.994,1.507,3.969,0.993
3,T1_interior,Avg Latency,s,dummy_median,18486,3120,3081,0.172,0.227,-0.244,0.174,0.230,-0.252
4,T1_interior,Avg Latency,s,linear,18486,3120,3081,0.079,0.098,0.767,0.083,0.104,0.744
5,T1_interior,Avg Latency,s,random_forest,18486,3120,3081,0.018,0.027,0.983,0.021,0.033,0.975
6,T1b_right_edge,Throughput,tx/s,dummy_median,18642,3120,2925,40.165,47.146,-0.005,40.206,47.165,-0.005
7,T1b_right_edge,Throughput,tx/s,linear,18642,3120,2925,5.290,7.113,0.977,5.497,7.231,0.976
8,T1b_right_edge,Throughput,tx/s,random_forest,18642,3120,2925,1.332,3.626,0.994,1.415,3.842,0.993
9,T1b_right_edge,Avg Latency,s,dummy_median,18642,3120,2925,0.172,0.227,-0.244,0.172,0.228,-0.278


## 3. Right-edge extrapolation reading (T1b): test MAE in physical units

*The code variable below is named `primary` from an earlier wording; T1b is not the primary split (see the introduction).*

In [4]:
primary = results_df[results_df['split']=='T1b_right_edge']
fig, axes = plt.subplots(1, 2, figsize=(13,4.6))
for ax, tgt, unit in zip(axes, TARGETS, ['tx/s','s']):
    sub = primary[primary['target']==tgt]
    ax.bar(sub['model'], sub['test_mae'], color=['#937860','#4C72B0','#55A868'])
    for x, m in zip(range(len(sub)), sub['test_mae']):
        ax.text(x, m, f'{m:.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_title(f'T1b test MAE - {tgt} ({unit})'); ax.set_ylabel(f'MAE ({unit})')
fig.suptitle('Baseline errors on the registered right-edge block-size holdout (block 710-800, trained max 605)')
fig.tight_layout(); fig.savefig(FIGS/'hfbtp_baseline_t1b_mae.png', dpi=150); plt.show()
primary[['target','model','test_mae','test_rmse','test_r2']].round(4)

C:\Users\mailf\AppData\Local\Temp\ipykernel_25744\2307284344.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIGS/'hfbtp_baseline_t1b_mae.png', dpi=150); plt.show()


,target,model,test_mae,test_rmse,test_r2
6,Throughput,dummy_median,40.2063,47.1653,-0.0048
7,Throughput,linear,5.4967,7.2314,0.9764
8,Throughput,random_forest,1.4149,3.8415,0.9933
9,Avg Latency,dummy_median,0.1720,0.2283,-0.2777
10,Avg Latency,linear,0.1201,0.1589,0.3813
11,Avg Latency,random_forest,0.0229,0.0419,0.9570


## 4. Actual-vs-predicted on T1b (Random Forest)

In [5]:
from fabric_perf.models import build_pipeline, separate_features_targets
tr, va, te, info = splits['T1b_right_edge']
fig, axes = plt.subplots(1, 2, figsize=(13,5))
for ax, tgt, unit in zip(axes, TARGETS, ['tx/s','s']):
    pipe = build_pipeline('random_forest')
    X_tr, y_tr = separate_features_targets(tr, target=tgt)
    X_te, y_te = separate_features_targets(te, target=tgt)
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    ax.scatter(y_te, pred, s=5, alpha=0.4, color='#4C72B0')
    lims = [min(y_te.min(), pred.min()), max(y_te.max(), pred.max())]
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set_xlabel(f'actual {tgt} ({unit})'); ax.set_ylabel(f'predicted {tgt} ({unit})')
    ax.set_title(f'RF on T1b test - {tgt}')
    ax.grid(alpha=.3)
fig.suptitle('Random Forest predictions vs actual on the registered block-size holdout')
fig.tight_layout(); fig.savefig(FIGS/'hfbtp_baseline_t1b_rf_pred_vs_actual.png', dpi=150); plt.show()

C:\Users\mailf\AppData\Local\Temp\ipykernel_25744\1968529501.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); fig.savefig(FIGS/'hfbtp_baseline_t1b_rf_pred_vs_actual.png', dpi=150); plt.show()


## 5. Where the simple models disagree with RF — per-band errors (T1b, Throughput, MLR — 8 features)

In [6]:
tr, va, te, info = splits['T1b_right_edge']
X_te, y_te = separate_features_targets(te, target='Throughput')
X_tr, y_tr = separate_features_targets(tr, target='Throughput')
lin = build_pipeline('linear'); lin.fit(X_tr, y_tr)
pred = lin.predict(X_te)
per_band = pd.DataFrame({'block': X_te['Block Size'].values, 'err': pred - y_te.values})
banded = per_band.groupby(pd.cut(per_band['block'], bins=[700,730,760,800]), observed=True)['err'].agg(['mean','std','count']).round(2)
banded
# Note: per the protocol, differences between models on a single-observation-per-cell dataset cannot be attributed to extrapolation with certainty (no noise floor).

,mean,std,count
block,,,
"(700, 730]",1.37,7.06,741
"(730, 760]",1.60,7.00,936
"(760, 800]",1.86,7.06,1248


## 6. Split-contrast summary table (experiment record)

In [7]:
summary = results_df[cols].round(4)
save_csv_rows(summary.to_dict(orient='records'), RES / 'hfbtp_baseline_metrics.csv')
run_record = {
    'executed_at_utc': utc_timestamp(),
    'seed': int(SEED),
    'python': platform.python_version(),
    'sklearn': sklearn.__version__, 'pandas': pd.__version__, 'numpy': np.__version__, 'matplotlib': matplotlib.__version__,
    'dataset_md5': check['md5'], 'dataset_sha256': check['sha256'],
    'feature_columns': FEATURE_COLUMNS,
    'raw_features': RAW_FEATURES,
    'targets': TARGETS,
    'models': ['dummy_median', 'linear', 'random_forest'],
    'random_forest_params': {'n_estimators': 200, 'random_state': 42, 'n_jobs': -1},
    'split_protocol': 'docs/evaluation_protocol.md (approved); T1b primary: test [710..800], val [610..705], train <=605',
    'split_definitions': {name: {k: v for k, v in info.items()} for name, (tr, va, te, info) in splits.items()},
    'interpretation_limits': 'Single observation per configuration cell: no noise floor, no repeatability estimate, no causal mechanism. Results are descriptive of the measured environment; they cannot establish controller benefit or real fabric deployment performance.',
}
save_json(run_record, RES / 'hfbtp_baseline_run_manifest.json')
print('saved:', RES / 'hfbtp_baseline_metrics.csv')
print('saved:', RES / 'hfbtp_baseline_run_manifest.json')
summary.round(3)

saved: C:\OJT_SEM3\AI-Assisted-Permissioned-Blockchain-Architecture-\experiments\results\hfbtp_baseline_metrics.csv
saved: C:\OJT_SEM3\AI-Assisted-Permissioned-Blockchain-Architecture-\experiments\results\hfbtp_baseline_run_manifest.json


,split,target,unit,model,train_rows,val_rows,test_rows,validation_mae,validation_rmse,validation_r2,test_mae,test_rmse,test_r2
0,T1_interior,Throughput,tx/s,dummy_median,18486,3120,3081,40.165,47.146,-0.005,40.119,47.073,-0.005
1,T1_interior,Throughput,tx/s,linear,18486,3120,3081,5.026,7.036,0.978,4.874,7.131,0.977
2,T1_interior,Throughput,tx/s,random_forest,18486,3120,3081,1.316,3.763,0.994,1.508,3.969,0.993
3,T1_interior,Avg Latency,s,dummy_median,18486,3120,3081,0.172,0.227,-0.244,0.174,0.230,-0.252
4,T1_interior,Avg Latency,s,linear,18486,3120,3081,0.079,0.098,0.767,0.083,0.104,0.744
5,T1_interior,Avg Latency,s,random_forest,18486,3120,3081,0.018,0.027,0.983,0.021,0.033,0.975
6,T1b_right_edge,Throughput,tx/s,dummy_median,18642,3120,2925,40.165,47.146,-0.005,40.206,47.165,-0.005
7,T1b_right_edge,Throughput,tx/s,linear,18642,3120,2925,5.290,7.113,0.977,5.497,7.231,0.976
8,T1b_right_edge,Throughput,tx/s,random_forest,18642,3120,2925,1.332,3.626,0.994,1.415,3.842,0.993
9,T1b_right_edge,Avg Latency,s,dummy_median,18642,3120,2925,0.172,0.227,-0.244,0.172,0.228,-0.278


## What this experiment does NOT establish

- **No practical-benefit claim:** beating a Dummy or a Multiple Linear Regression baseline in R² does not show that a predictive controller outperforms static or reactive control in a live Fabric network. That requires closed-loop experiments (out of scope for this phase).
- **No uncertainty statement:** one observation per cell means MAE/RMSE here average over mean response error across configurations, without any run-to-run noise floor — model ranking near 0.5-1 tx/s MAE differences is not resolvable.
- **No extrapolation guarantee:** T1b measures right-edge behavior inside the measured grid; it cannot certify behavior beyond block size 800 or on other hardware/Fabric versions.